# Episode 18: Research with a Feedback Loop

Real work isn't one-and-done. A draft gets reviewed, revised, reviewed again. This episode builds that **feedback loop** on a `workflow` channel.

**In this episode you'll build:** a drafter and a reviewer that iterate on a tagline until it's approved — or a turn cap stops them.

## The two ways a loop ends

A feedback loop alternates two agents. The interesting question is *when it stops*. A `workflow` graph gives you two termination routes, and a robust loop uses **both**:

- **Approval** — the reviewer calls an `approve` tool that sets a channel **context variable**; a `ContextEquals("done", True)` transition then terminates the channel. This is the "we're satisfied" exit.
- **A turn cap** — `max_turns` on the graph is a safety net, so a loop that never converges still ends.

This episode is really about **context variables** — channel-scoped state that a tool writes and a transition reads.

## Setup

We need workflow imports plus `ContextEquals` (the condition), `ChannelInject` (to reach the channel from a tool), and `set_context` (to write a context variable).

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import asyncio

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.knowledge import MemoryKnowledgeStore
from autogen.beta.network import (
    AgentTarget,
    ChannelInject,
    ContextEquals,
    EV_CHANNEL_CLOSED,
    EV_PACKET,
    EV_TEXT,
    FromSpeaker,
    Hub,
    HubClient,
    LocalLink,
    Passport,
    Resume,
    TerminateTarget,
    Transition,
    TransitionGraph,
)
from autogen.beta.network.workflow_helpers import set_context

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


async def wait_for_close(hub, channel_id, timeout=240.0):
    deadline = asyncio.get_event_loop().time() + timeout
    while asyncio.get_event_loop().time() < deadline:
        for e in await hub.read_wal(channel_id):
            if e.event_type == EV_CHANNEL_CLOSED:
                return e
        await asyncio.sleep(0.1)
    raise asyncio.TimeoutError("no close")


def turn_text(env):
    if env.event_type == EV_TEXT:
        return env.event_data.get("text")
    if env.event_type == EV_PACKET:
        return env.event_data.get("body")
    return None

## The drafter, the reviewer, and the `approve` tool

The reviewer gets an `approve` tool. When called, it writes `done=True` into the channel's context via `set_context` — and the graph watches for exactly that.

In [ ]:
hub = await Hub.open(MemoryKnowledgeStore(), ttl_sweep_interval=0)
link = LocalLink(hub)
u_hc, d_hc, r_hc = (HubClient(link, hub=hub) for _ in range(3))

user = await u_hc.register_human(Passport(name="user", kind="human"))

drafter_agent = Agent(
    "drafter",
    prompt="You write product taglines. Read the brief and the latest reviewer "
    "feedback (if any) and output ONE improved tagline only.",
    config=config,
)

reviewer_agent = Agent(
    "reviewer",
    prompt="You review a tagline. Look at the conversation so far. "
    "If the drafter has revised the tagline at least once already, call the "
    "approve tool and reply 'Approved.' Otherwise, reply with exactly one short "
    "instruction to improve it and do NOT call approve.",
    config=config,
)


@reviewer_agent.tool
async def approve(channel: ChannelInject) -> str:
    """Approve the current tagline and end the review loop."""
    if channel is None:
        return "no channel"
    await set_context(channel, "done", True)  # the graph watches this
    return "approved"


drafter = await d_hc.register(
    drafter_agent, Passport(name="drafter"), Resume(), attach_plugin=False
)
reviewer = await r_hc.register(
    reviewer_agent, Passport(name="reviewer"), Resume(), attach_plugin=False
)
print("drafter + reviewer registered")

## The loop graph

The graph alternates drafter → reviewer → drafter → reviewer. The **first** transition checks `ContextEquals("done", True)` — the approval exit. `max_turns=6` is the safety cap.

In [1]:
graph = TransitionGraph(
    initial_speaker=user.agent_id,
    transitions=[
        # Approval exit -- listed FIRST so it wins as soon as `done` is set.
        Transition(
            when=ContextEquals("done", value=True), then=TerminateTarget("approved")
        ),
        Transition(when=FromSpeaker(user.agent_id), then=AgentTarget(drafter.agent_id)),
        Transition(
            when=FromSpeaker(drafter.agent_id), then=AgentTarget(reviewer.agent_id)
        ),
        Transition(
            when=FromSpeaker(reviewer.agent_id), then=AgentTarget(drafter.agent_id)
        ),
    ],
    default_target=TerminateTarget("max_iterations"),
    max_turns=6,  # safety net if the reviewer never approves
)
channel = await user.open(
    type="workflow",
    target=[drafter.agent_id, reviewer.agent_id],
    knobs={"graph": graph.to_dict()},
)
await channel.send("Brief: a tagline for a fast note-taking app.")

close_env = await wait_for_close(hub, channel.channel_id)
print(f"closed: {close_env.event_data.get('reason')!r}")

names = {
    user.agent_id: "user",
    drafter.agent_id: "drafter",
    reviewer.agent_id: "reviewer",
}
for env in await hub.read_wal(channel.channel_id):
    text = turn_text(env)
    if text and text.strip():
        print(f"[{names[env.sender_id]}] {text}")

for hc in (u_hc, d_hc, r_hc):
    await hc.close()
await hub.close()

closed: 'max_turns'
[user] Brief: a tagline for a fast note-taking app.
[drafter] Capture thoughts at the speed of inspiration.
[reviewer] Make the tagline more concise and catchy.
[drafter] Capture ideas. Fast. Simple.
[reviewer] Make the tagline more specific to note-taking to clearly convey the app's purpose.
[drafter] Capture thoughts fast, note smarter.


## Reading the run

Watch the tagline genuinely improve across the loop: *"Capture thoughts at the speed of inspiration."* → *"Capture ideas. Fast. Simple."* → *"Capture thoughts fast, note smarter."* Each pass folds in the reviewer's note.

This run closed with `max_turns` — the reviewer kept finding something to refine, so the **safety net** fired after 6 turns. Had the reviewer called `approve`, `set_context` would have written `done=True`, the `ContextEquals` transition would have matched, and the channel would have closed with `approved` instead.

Both exits are real and both matter: `ContextEquals` is the *intended* stop; `max_turns` guarantees the loop is *bounded* regardless.

## Additional content

**Context variables.** `set_context(channel, key, value)` writes channel-scoped state onto the WAL; `ContextEquals(key, value)` reads it in a transition. That write/read pair is the general mechanism for context-driven routing — far beyond approve/reject.

**Order matters.** The `ContextEquals` transition is listed *first*. The adapter takes the first matching transition, so the approval check must come before the `FromSpeaker` rules that would otherwise just continue the loop.

**Parallel research.** A full research assistant pairs this loop with *parallel* fact-gathering — `TaskConfig` + `run_subtasks(parallel=True)` from Episode 4 — researching several sub-questions at once, then looping a writer and reviewer over the synthesis.

## What's next

The agents here worked from a brief alone. Episode 19 gives an agent a **knowledge base** — durable memory it can read from and write to.